# 04 · Evaluación y explicabilidad (OE3)

Fase 5 de CRISP-DM. Optimiza los hiperparámetros de los cuatro modelos que pasaron la Fase 4, selecciona el modelo final, fija el umbral de decisión con el entrenamiento y evalúa **una sola vez** en el conjunto de prueba. Al final explica las predicciones con valores SHAP.

**Tiempo estimado en Colab: entre 10 y 25 minutos.** No cierres la pestaña mientras corre.

In [ ]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import os, warnings
os.environ["PYTHONWARNINGS"] = "ignore:Unknown solver options"  # también en los procesos paralelos
warnings.filterwarnings("ignore", message="Unknown solver options")  # aviso interno de SciPy, no afecta resultados

import pandas as pd
from src import config, plots, prep, evaluation as ev, explain
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
df = prep.add_no_enrollment_flags(binarize_target(load_raw()))
idx_train, idx_test = prep.load_split()
y_test = df.loc[idx_test, config.BINARY_TARGET]
print("Entrenamiento:", len(idx_train), "| Prueba:", len(idx_test))

## 1. Optimización de hiperparámetros

Búsqueda en malla para la regresión logística y búsqueda aleatoria para Random Forest, XGBoost y SVM, con AUC-ROC en validación cruzada estratificada de 5 pliegues sobre el entrenamiento. La ponderación de clases entra como un hiperparámetro más (con o sin ponderación). Momento: cierre del 2.º semestre.

In [ ]:
optimizacion, mejores = ev.tune(df, idx_train)
optimizacion.to_csv(config.REPORTS / "optimizacion.csv", index=False, encoding="utf-8-sig")
optimizacion[["modelo", "auc_cv", "auc_de", "auc_ee", "combinaciones", "variables"]].round(4)

## 2. Selección del modelo final

Gana el mayor AUC-ROC. Criterio de parsimonia fijado de antemano: si la regresión logística queda a menos de un error estándar del mejor, se elige la regresión logística por ser la más simple e interpretable.

In [ ]:
modelo_final = ev.select(optimizacion)
parametros = mejores[modelo_final]
print("Modelo final:", modelo_final)
print("Hiperparámetros:", parametros)

## 3. Umbral de decisión y evaluación en los tres momentos

Para cada momento: el umbral se elige con probabilidades fuera de pliegue **del entrenamiento** (el más alto que mantiene una sensibilidad de al menos 0,80), luego el modelo se entrena con todo el entrenamiento y se evalúa una única vez en la prueba.

In [ ]:
momentos, salidas = ev.evaluate_moments(df, idx_train, idx_test, modelo_final, parametros)
momentos.to_csv(config.REPORTS / "evaluacion_momentos.csv", index=False, encoding="utf-8-sig")
especificacion = ev.save_final(modelo_final, parametros, salidas)
momentos.round(3)

## 4. Evaluación detallada al cierre del 2.º semestre

Métricas con el umbral elegido y con el umbral convencional de 0,5, intervalos de confianza del 95 % por bootstrap (1.000 remuestreos de la prueba) y verificación de los criterios de éxito.

In [ ]:
s2 = salidas["semestre_2"]
p_test, umbral = s2["p_test"], s2["umbral"]
evaluacion = pd.DataFrame([ev.metrics(y_test, p_test, umbral), ev.metrics(y_test, p_test, 0.5)],
                          index=["umbral elegido", "umbral 0,5"])
evaluacion.to_csv(config.REPORTS / "evaluacion_prueba.csv", encoding="utf-8-sig")
ic = pd.DataFrame(ev.bootstrap_ci(y_test, p_test, umbral), index=["ic95_inferior", "ic95_superior"]).T
ic.to_csv(config.REPORTS / "intervalos_confianza.csv", encoding="utf-8-sig")
criterios = ev.success_table(evaluacion.loc["umbral elegido"])
criterios.to_csv(config.REPORTS / "criterios_exito.csv", index=False, encoding="utf-8-sig")
plots.roc_calibration(y_test, p_test, modelo_final)
plots.confusion(evaluacion.loc["umbral elegido"])
display(evaluacion.round(3)); display(ic.round(3)); criterios

## 5. Explicabilidad con valores SHAP

Se explican las predicciones del modelo final sobre el conjunto de prueba. La importancia global suma los aportes de las columnas one-hot de cada variable original. Se compara con el tamaño del efecto de la Fase 2 (correlación de Spearman) para verificar coherencia.

In [ ]:
pipe, cols = s2["pipeline"], s2["cols"]
X_test = df.loc[idx_test, cols]
casos = explain.pick_cases(y_test, p_test, umbral)
filas = X_test.index
if modelo_final == "SVM":   # el explicador por permutaciones es lento: muestra de 200 + los casos
    filas = X_test.sample(200, random_state=config.SEED).index.union(pd.Index(list(casos.values())))
exp, Xt = explain.shap_values(pipe, df.loc[idx_train, cols], X_test.loc[filas])
omap = explain.origin_map(df, cols, exp.feature_names)
etiquetas = [explain.readable(f, omap[f]) for f in exp.feature_names]

importancia = explain.global_importance(exp, df, cols)
importancia.to_csv(config.REPORTS / "shap_importancia.csv", index=False, encoding="utf-8-sig")
rho, n_var = explain.coherence_with_oe1(importancia)
print(f"Coherencia con la Fase 2: rho de Spearman = {rho:.3f} sobre {n_var} variables")

plots.shap_global(importancia)
plots.shap_beeswarm(exp, etiquetas)
pos = {idx: i for i, idx in enumerate(Xt.index)}
for nombre, idx in casos.items():
    archivo = "fig_12_shap_caso_detectado" if "no" not in nombre else "fig_13_shap_caso_no_detectado"
    print(nombre, "| índice", idx, "| riesgo", round(float(pd.Series(p_test, index=y_test.index)[idx]), 3))
    plots.shap_case(exp[pos[idx]], etiquetas, archivo,
                    raw_values=explain.raw_row(X_test.loc[idx], exp.feature_names, omap))
importancia.head(15).round(4)

## 6. Resumen para el documento

Ejecutar esta celda y enviar su salida completa.

In [ ]:
print("OPTIMIZACIÓN (AUC-ROC en validación cruzada, 2.º semestre):")
print(optimizacion.round(4).to_string(index=False))
print("\nMODELO FINAL:", modelo_final, "|", parametros)
print("\nEVALUACIÓN EN PRUEBA POR MOMENTO:")
print(momentos.round(3).to_string(index=False))
print("\nEVALUACIÓN DETALLADA 2.º SEMESTRE:")
print(evaluacion.round(3).to_string())
print("\nINTERVALOS DE CONFIANZA 95 % (bootstrap):")
print(ic.round(3).to_string())
print("\nCRITERIOS DE ÉXITO:")
print(criterios.round(3).to_string(index=False))
print(f"\nSHAP: coherencia con la Fase 2, rho = {rho:.3f} ({n_var} variables)")
print(importancia.head(15).round(4).to_string(index=False))
for nombre, idx in casos.items():
    i = pos[idx]
    top = pd.Series(exp.values[i], index=etiquetas).sort_values(key=abs, ascending=False).head(5)
    print(f"\nCASO {nombre.upper()} (riesgo {pd.Series(p_test, index=y_test.index)[idx]:.3f}):")
    print(top.round(3).to_string())

## 7. Guardar resultados en GitHub

La siguiente celda empaqueta la carpeta `reports/` en `reports_fase5.zip`. Descárgalo, descomprímelo y sube su contenido a la carpeta `reports` del repositorio con el mensaje `Fase 5: evaluación y SHAP`. Luego guarda este cuaderno con Archivo > Guardar una copia en GitHub.

In [ ]:
!cd /content/tesis-desercion && zip -qr /content/reports_fase5.zip reports && echo "Listo: /content/reports_fase5.zip"